## 📖 How to use this notebook

- **Open it in Google Colab** and choose `File → Save a copy in Drive` first, so your work is kept.
- **Run a cell** by clicking on it and pressing **Shift + Enter** (or the ▶ button on its left). **Run the cells in order, top to bottom** — later cells use results from earlier ones.
- **✏️ TODO** cells contain a blank written as `___`. Replace the `___` (underscores included) with your answer, then run the cell. Each blank is one small thing — usually a single word or number.
- **💡 Hint** lines in the TODO cell tell you what to type.
- **✅ Stuck?** Right below every TODO there is a hidden answer. Click it to reveal. Using it is completely fine — the goal is to understand what the code does, not to type it from memory.
- **⭐ Optional** sections are extras for if you finish early or want to explore after the workshop. You can skip them; nothing later in the notebook depends on them.
- **Lines starting with `#` are comments** written for you to read. Python ignores them.
- **If something goes wrong:** `Runtime → Restart session`, then run the cells from the top again. Or raise your hand — that's what the TAs are for.

No coding background is needed.

# Session 1 — Data Mining, APIs & Web Scraping for Bioinformatics


**Today's goal:** learn the main ways to get biological data into Python — reading sequence files, asking online databases (APIs), pulling tables out of web pages, and reading PDFs — and turn the result into a clean table you can analyse.

| Part | Topic | Time |
|---|---|---|
| Setup | Install and import the tools | ~3 min |
| 1 | Parse a FASTA file | ~10 min |
| 2 | Query live biological APIs (NCBI, UniProt, RNAcentral) | ~20 min |
| 3 | Find and use the RCSB PDB search API | ~10 min |
| 4 | Scrape a table from a web page | ~10 min |
| 5 | ⭐ Optional — Headless browser automation (Selenium) | ~10 min |
| 6 | Capstone: build a reusable data pipeline | ~10 min |
| 7 | ⭐ Optional — Extract text & tables from a PDF (Docling) | ~10 min |

---
## Setup — run this first ⏱ ~3 min

This installs the Python libraries we need (it may take a minute) and loads them:
- **Biopython** — reading biological sequence files
- **requests** — talking to websites and APIs
- **pandas** — tables (like Excel, but in Python)
- **BeautifulSoup** — reading web pages (HTML)

When it prints `Setup complete.` you're ready.

In [1]:
!pip install -q biopython requests beautifulsoup4 lxml pandas pypdf
import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 80)
print("Setup complete.")

Setup complete.


---
## Part 1 — Parse Your First FASTA File ⏱ ~10 min
*(companion to slide: "Parse Your First FASTA File")*

**FASTA** is the most common text format for biological sequences. Every record has two parts:
- a **header line** starting with `>` — the sequence ID followed by a description
- one or more lines of **sequence**

The cell below writes a tiny FASTA file with three human microRNA precursors to disk. It is written inline, so it works even without internet. In Part 6 you'll pull records live from RNAcentral instead.

👉 **Just run it.**

In [ ]:
fasta_text = """>hsa-let-7a-1 MI0000060 Homo sapiens let-7a-1
UGGGAUGAGGUAGUAGGUUGUAUAGUUUUAGGGUCACACCCACCACUGGGAGAUAACUAUACAAUCUACUGUCUUUCCUA
>hsa-mir-21 MI0000077 Homo sapiens mir-21
UGUCGGGUAGCUUAUCAGACUGAUGUUGACUGUUGAAUCUCAUGGCAACACCAGUCGAUGGGCUGUCUGACA
>hsa-mir-155 MI0000681 Homo sapiens mir-155
CUGUUAAUGCUAAUUGUGAUAGGGGUUUUUGCCUCCAACUGACUCCUACAUAUUAGCAUUAACAG
"""

with open("sample_mirna.fasta", "w") as f:
    f.write(fasta_text)

print(fasta_text)

### 1.1 Read the file with Biopython

`SeqIO.parse(filename, "fasta")` reads a sequence file and gives back one **record** per sequence. Each record has:
- `rec.id` — the ID (first word of the header)
- `rec.description` — the whole header line
- `rec.seq` — the sequence itself

**Expected result:** three lines, one per microRNA, with lengths 80, 72 and 65.

In [3]:
from Bio import SeqIO

# ✏️ TODO: Read our FASTA file. Replace the blank with the file name, in quotes.
#          ("fasta" tells Biopython what format the file is in — that part is done for you.)
# 💡 Hint: the file name is "sample_mirna.fasta" — it was created in the cell above.
records = list(SeqIO.parse(___, "fasta"))

for rec in records:
    print(f"ID: {rec.id:20s}  length: {len(rec.seq):3d}  description: {rec.description}")

FileNotFoundError: [Errno 2] No such file or directory: ''

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
records = list(SeqIO.parse("sample_mirna.fasta", "fasta"))
```

</details>

### 1.2 Turn the records into a table

A **pandas DataFrame** is Python's version of a spreadsheet: rows plus named columns. The standard recipe for building one is:
1. start with an empty list (`rows = []`)
2. for every record, add a **dictionary** — `{"column name": value, ...}` — to the list
3. hand the finished list to `pd.DataFrame(rows)`

You will see this exact recipe again several times today.

In [ ]:
# ✏️ TODO: Two columns are already filled in. Fill in the two blanks so each row also
#          stores the sequence and its length.
# 💡 Hint: the sequence is   str(rec.seq)    (str turns it into plain text)
# 💡 Hint: its length is     len(rec.seq)

rows = []
for rec in records:
    rows.append({
        "id": rec.id,
        "description": rec.description,
        "sequence": ___,
        "length": ___,
    })

mirna_df = pd.DataFrame(rows)
mirna_df

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
"sequence": str(rec.seq),
"length": len(rec.seq),
```

</details>

---
## Part 2 — Query Live Biological APIs ⏱ ~20 min
*(companion to slide: "Query Live Biological APIs")*

An **API** is a web address that returns data for *programs* rather than a web page for *people*. Instead of clicking through a website, you send a request and get structured data back.

In Python this is always the same two steps:
1. `requests.get(url, params=...)` — ask the server for data
2. `.json()` — turn the reply into a Python **dictionary** you can look things up in

We'll use three public APIs that need no key or login:
- **NCBI E-utilities** — genes, sequences, literature · docs: ncbi.nlm.nih.gov/books/NBK25501
- **UniProt REST API** — proteins · interactive docs: rest.uniprot.org/docs
- **RNAcentral REST API** — non-coding RNAs · docs: rnacentral.org/api

💡 Before writing any code, it is worth glancing at an API's documentation. Many APIs (including UniProt's) publish an **OpenAPI** description, which powers an interactive "try it in the browser" page. It tells you which parameters exist and what the answer looks like before you write a single line of Python.

### 2.1 NCBI — search for a gene and fetch its summary

NCBI works in two steps:
- **esearch** — "which IDs match my search?" → returns a list of IDs
- **esummary** — "tell me about this ID" → returns the details

We'll search for the human gene **MIR21**, which encodes microRNA-21.

In [ ]:
ncbi_base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# Step 1: describe our search as a dictionary of parameters
search_params = {
    "db": "gene",                           # which NCBI database
    "term": "MIR21[gene] AND human[orgn]",  # what to search for
    "retmode": "json",                      # give us the answer as JSON
}

# ✏️ TODO: Send the search. Replace the blank with the name of the parameter dictionary above.
# 💡 Hint: it is called  search_params
search_resp = requests.get(f"{ncbi_base}/esearch.fcgi", params=___, timeout=30)
search_resp.raise_for_status()      # stops with a clear error message if the request failed

search_json = search_resp.json()    # turn the reply into a Python dictionary
gene_ids = search_json["esearchresult"]["idlist"]
print("Gene IDs found:", gene_ids)

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
search_resp = requests.get(f"{ncbi_base}/esearch.fcgi", params=search_params, timeout=30)
```

</details>

⭐ **Optional peek:** run the cell below to see the raw dictionary NCBI sent back. Can you find `idlist` in it? That is where the line `search_json["esearchresult"]["idlist"]` pulled the IDs from.

In [ ]:
search_json

**Step 2:** now ask for the details of the first ID we found. The request is identical in shape — only the endpoint (`esummary`) and the parameters change.

In [ ]:
summary_params = {"db": "gene", "id": gene_ids[0], "retmode": "json"}

summary_resp = requests.get(f"{ncbi_base}/esummary.fcgi", params=summary_params, timeout=30)
summary_resp.raise_for_status()

# ✏️ TODO: Turn the reply into a dictionary.
# 💡 Hint: in Step 1 we wrote  search_resp.json()  — here the reply is called  summary_resp
summary_json = ___

gene_info = summary_json["result"][gene_ids[0]]
print("Name:       ", gene_info.get("name"))
print("Description:", gene_info.get("description"))
print("Organism:   ", gene_info.get("organism", {}).get("scientificname"))
print("Chromosome: ", gene_info.get("chromosome"))

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
summary_json = summary_resp.json()
```

</details>

⭐ **Optional peek:** the full summary record. Scroll through — there is much more in it than the four fields we printed.

In [ ]:
summary_json

### 2.2 UniProt — fetch a protein record

UniProt identifies every protein with an **accession**. Here we use **P05067**, the human amyloid-beta precursor protein (APP). Adding `.json` to the end of the URL asks UniProt for machine-readable data instead of a web page.

**Expected result:** the protein name, the gene `APP`, and a length of 770 amino acids.

In [ ]:
uniprot_url = "https://rest.uniprot.org/uniprotkb/P05067.json"   # human APP protein

# ✏️ TODO: Request the URL above. Replace the blank with the variable that holds the URL.
# 💡 Hint: uniprot_url
resp = requests.get(___, timeout=30)
resp.raise_for_status()
data = resp.json()

print("Protein name:", data["proteinDescription"]["recommendedName"]["fullName"]["value"])
print("Gene(s):     ", [g["geneName"]["value"] for g in data.get("genes", [])])
print("Sequence len:", data["sequence"]["length"])

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
resp = requests.get(uniprot_url, timeout=30)
```

</details>

🧪 **Try it:** change `P05067` in the URL to another accession and re-run — for example `P69905` (hemoglobin subunit alpha) or `P04637` (p53).

### 2.3 RNAcentral — search for non-coding RNAs

RNAcentral collects non-coding RNA sequences from many databases. We send a search query and get back a list called `results` — one dictionary per RNA record. This cell just downloads the data; the TODO comes next.

In [ ]:
rnacentral_url = "https://rnacentral.org/api/v1/rna"
params = {"query": "let-7 AND TAXONOMY:9606", "format": "json", "page_size": 10}

resp = requests.get(rnacentral_url, params=params, timeout=60)
resp.raise_for_status()
results = resp.json()["results"]     # a list: one dictionary per RNA record
print(f"Downloaded {len(results)} records")

Now build a table from `results`, using the same list-of-dictionaries recipe as in Part 1.2.

`r.get("key")` looks up a value in a dictionary. Unlike `r["key"]`, it returns `None` instead of crashing if the key happens to be missing — useful with real-world data.

In [ ]:
# ✏️ TODO: Two columns are done for you. Fill in the blanks for the other two.
# 💡 Hint: follow the same pattern:  r.get("length")  and  r.get("rna_type")
rows = []
for r in results:
    rows.append({
        "rnacentral_id": r.get("rnacentral_id"),
        "description": r.get("description"),
        "length": ___,
        "rna_type": ___,
    })

rnacentral_df = pd.DataFrame(rows)
rnacentral_df

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
"length": r.get("length"),
"rna_type": r.get("rna_type"),
```

</details>

⭐ **Optional peek:** the raw records. Each one has more fields than we kept (`sequence`, `md5`, links to cross-references…).

In [ ]:
results

Save the table to a **CSV** file (a plain-text spreadsheet that Excel, R and Python can all open). This is our first real structured dataset. You can see the file in Colab's 📁 **Files** panel on the left.

In [ ]:
rnacentral_df.to_csv("rnacentral_let7_search.csv", index=False)
print("Saved rnacentral_let7_search.csv with", len(rnacentral_df), "rows")

---
## Part 3 — Find & Hit RCSB PDB's Search API ⏱ ~10 min
*(companion to slide: "Find & Hit RCSB PDB's Search API")*

Many websites use an API behind the scenes even when they don't advertise one. If you can find it, you can use it directly from Python — no scraping needed.

**This part starts in the browser.** Open this URL in Chrome:

```
https://www.rcsb.org/search?request=%7B%22query%22%3A%7B%22type%22%3A%22group%22%2C%22logical_operator%22%3A%22and%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22group%22%2C%22logical_operator%22%3A%22and%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22group%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22terminal%22%2C%22service%22%3A%22text%22%2C%22parameters%22%3A%7B%22attribute%22%3A%22rcsb_entry_container_identifiers.entry_id%22%2C%22operator%22%3A%22in%22%2C%22negation%22%3Afalse%2C%22value%22%3A%5B%221b23%22%5D%7D%7D%5D%2C%22logical_operator%22%3A%22and%22%7D%5D%2C%22label%22%3A%22text%22%7D%5D%7D%2C%22return_type%22%3A%22entry%22%2C%22request_options%22%3A%7B%22paginate%22%3A%7B%22start%22%3A0%2C%22rows%22%3A25%7D%2C%22results_content_type%22%3A%5B%22experimental%22%5D%2C%22sort%22%3A%5B%7B%22sort_by%22%3A%22score%22%2C%22direction%22%3A%22desc%22%7D%5D%2C%22scoring_strategy%22%3A%22combined%22%7D%2C%22request_info%22%3A%7B%22query_id%22%3A%22aff2fd0e19262ac90435ea4238c137b1%22%7D%7D
```

1. Right-click anywhere on the page → **Inspect** → open the **Network** tab and filter by **Fetch/XHR**. Reload the page and watch the requests the page makes.
2. **Console tab:** select every result checkbox in one line. Open the **Console** tab and run:
   ```js
   document.querySelectorAll('input[type="checkbox"]').forEach(cb => cb.checked = true);
   ```
   The Console is a live JavaScript environment for the page, not just a viewer — you can change the page directly, no Python required.

**Back in Python:** notice that the page URL itself already contains the search, in the `request=` part. The `%7B`, `%22` etc. are just "URL-encoded" characters (`{` and `"`). Let's decode it.

In [ ]:
import json
from urllib.parse import urlparse, parse_qs, unquote

page_url = (
    "https://www.rcsb.org/search?request=%7B%22query%22%3A%7B%22type%22%3A%22group%22%2C"
    "%22logical_operator%22%3A%22and%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22group%22%2C"
    "%22logical_operator%22%3A%22and%22%2C%22nodes%22%3A%5B%7B%22type%22%3A%22group%22%2C"
    "%22nodes%22%3A%5B%7B%22type%22%3A%22terminal%22%2C%22service%22%3A%22text%22%2C"
    "%22parameters%22%3A%7B%22attribute%22%3A%22rcsb_entry_container_identifiers.entry_id%22%2C"
    "%22operator%22%3A%22in%22%2C%22negation%22%3Afalse%2C%22value%22%3A%5B%221b23%22%5D%7D%7D%5D%2C"
    "%22logical_operator%22%3A%22and%22%7D%5D%2C%22label%22%3A%22text%22%7D%5D%7D%2C"
    "%22return_type%22%3A%22entry%22%2C%22request_options%22%3A%7B%22paginate%22%3A%7B%22start%22%3A0%2C"
    "%22rows%22%3A25%7D%2C%22results_content_type%22%3A%5B%22experimental%22%5D%2C"
    "%22sort%22%3A%5B%7B%22sort_by%22%3A%22score%22%2C%22direction%22%3A%22desc%22%7D%5D%2C"
    "%22scoring_strategy%22%3A%22combined%22%7D%2C%22request_info%22%3A%7B%22query_id%22%3A"
    "%22aff2fd0e19262ac90435ea4238c137b1%22%7D%7D"
)

Two tools do the decoding:
- `urlparse(page_url).query` cuts out everything after the `?` in the URL
- `parse_qs(...)` splits that into a dictionary of parameters, and decodes the `%xx` characters
- `json.loads(...)` turns JSON **text** into a Python **dictionary**

In [ ]:
# ✏️ TODO (blank 1): split the URL into its parameters.
# 💡 Hint: urlparse(page_url).query
query_params = parse_qs(___)

# ✏️ TODO (blank 2): turn the "request" parameter (which is JSON text) into a dictionary.
# 💡 Hint: query_params["request"][0]     ([0] because parse_qs always gives a list)
decoded_request = json.loads(___)

print(json.dumps(decoded_request, indent=2))   # pretty-print it

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
query_params = parse_qs(urlparse(page_url).query)
decoded_request = json.loads(query_params["request"][0])
```

</details>

That decoded JSON **is** the exact request the page's JavaScript sends to `https://search.rcsb.org/rcsbsearch/v2/query` — the same one you'd see in the Network tab. No guessing required; the URL handed it to us.

To use it we **POST** it (send data *to* the server) instead of GET-ing a URL. With `requests.post(url, json=...)`, Python converts the dictionary to JSON for us.

In [ ]:
search_api_url = "https://search.rcsb.org/rcsbsearch/v2/query"

# ✏️ TODO: Send our decoded search to the RCSB Search API.
# 💡 Hint: the dictionary we just decoded is called  decoded_request
resp = requests.post(search_api_url, json=___, timeout=30)
print("Status code:", resp.status_code)    # 200 means success
resp.json()

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
resp = requests.post(search_api_url, json=decoded_request, timeout=30)
```

</details>

🗣️ **Discussion:** compare this to scraping the HTML of the rendered search results page instead. Which approach is faster? Which is more likely to break if RCSB redesigns their website?

---
## Part 4 — Scrape a Table with BeautifulSoup ⏱ ~10 min
*(companion to slide: "Scrape a Table with BeautifulSoup")*

When a website has **no API**, you can still read the page itself. Web pages are written in **HTML**, which is built from nested *tags*. A table looks like this:

```html
<table class="wikitable">
  <tr> <th>Amino acid</th> <th>Codons</th> </tr>   ← header row
  <tr> <td>Ala</td>        <td>GCU, GCC…</td> </tr>  ← data row
</table>
```
`<tr>` = table row, `<th>` = header cell, `<td>` = data cell. **BeautifulSoup** lets us search the page for these tags.

We'll use a Wikipedia page (the genetic code tables) as a stable, always-available example. The same technique works on any biological database page that lacks an API.

In [ ]:
wiki_url = "https://en.wikipedia.org/wiki/DNA_and_RNA_codon_tables"

# Wikipedia asks scripts to identify themselves, so we send a "User-Agent" header
resp = requests.get(wiki_url, headers={"User-Agent": "RNA-Canada-Workshop/1.0"}, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "lxml")      # parse the HTML so we can search it

# ✏️ TODO: Find every table on the page. Replace the blank with the HTML tag name, in quotes.
# 💡 Hint: the tag for a table is "table"
tables = soup.find_all(___, {"class": "wikitable"})
print(f"Found {len(tables)} wikitable(s) on the page")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
tables = soup.find_all("table", {"class": "wikitable"})
```

</details>

Now pull the text out of one table. The loop below goes through every row (`<tr>`) and collects the text of every cell in it — this part is done for you. Your job is the last step: turning the collected rows into a DataFrame.

In [ ]:
table = tables[2]      # the third table on the page (Python starts counting at 0)

# the first row holds the column names
header = [cell.get_text(strip=True) for cell in table.find_all("tr")[0].find_all(["th", "td"])]

# every other row holds data
rows = []
for tr in table.find_all("tr")[1:]:
    cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
    if cells:
        rows.append(cells)

# ✏️ TODO: Turn the list of rows into a DataFrame.
# 💡 Hint: the list of rows is called  rows
#    (header[:len(rows[0])] just makes sure we have exactly one name per column)
scraped_df = pd.DataFrame(___, columns=header[:len(rows[0])])
scraped_df.head(10)

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
scraped_df = pd.DataFrame(rows, columns=header[:len(rows[0])])
```

</details>

Real scraped data is messy. Wikipedia adds reference markers such as `[1]` to cells. The cell below removes them with a **regular expression** (a text pattern: `\[\d+\]` means "a `[`, one or more digits, then a `]`") and saves the clean table.

In [ ]:
import re

def clean_cell(x):
    if isinstance(x, str):
        return re.sub(r"\[\d+\]", "", x).strip()
    return x

# .map applies clean_cell to every cell (older pandas versions call this .applymap)
scraped_df_clean = scraped_df.map(clean_cell) if hasattr(scraped_df, "map") else scraped_df.applymap(clean_cell)
scraped_df_clean.to_csv("scraped_table.csv", index=False)
scraped_df_clean.head(10)

---
## ⭐ Optional — Part 5: Headless Selenium in Colab ⏱ ~10 min
*(companion to slide: "Headless Selenium in Colab")*

> **Optional:** the instructor will demo this live. The install below takes **2–4 minutes**, so feel free to just watch, and try it yourself after the workshop. Nothing later depends on it.

Some websites only show their data after you click, type or scroll — the content is generated by JavaScript, so `requests` sees an empty page. **Selenium** drives a real web browser from Python to handle those cases.

The instructor drives a **visible** browser. Here you run the same kind of automation **headless** — no window, identical behaviour. Colab needs this because there is no screen attached to the notebook.

**Step 1 — install Google Chrome and Selenium inside Colab** (just run it; ignore the wall of installation text).

In [ ]:
# 1. Add Google's official signing key and repository to Colab
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list'

# 2. Update package lists and install real Google Chrome + Selenium quietly
!apt-get update -y -q
!apt-get install -y -q google-chrome-stable
!pip install -q selenium

**Step 2 — drive the browser:** open Wikipedia's search page, type a query, press Enter, wait for the article to load, and read its first paragraph. Read the printed steps as it runs.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. Setup Headless Chrome
options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

try:
    print("1. Navigating to Wikipedia's Search Page...")
    driver.get("https://en.wikipedia.org/wiki/Special:Search")

    print("2. Entering query and hitting 'ENTER' simultaneously...")
    search_box = wait.until(EC.element_to_be_clickable((By.NAME, "search")))

    # THE FIX: Send the text and the return key in one single action!
    # This prevents the DOM from refreshing before the submit happens.
    search_box.send_keys("Ribonucleic acid", Keys.RETURN)

    print("3. Waiting for the official RNA article to load...")
    title = wait.until(EC.presence_of_element_located((By.ID, "firstHeading")))

    print("4. Scraping the scientific summary...")
    paragraphs = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".mw-parser-output p")))

    print("-" * 60)
    print(f"DEMO SUCCESSFUL! Landed on: {title.text}")
    print("-" * 60)

    for p in paragraphs:
        text = p.text.strip()
        if len(text) > 100:
            print("First Paragraph:")
            print(text)
            break

finally:
    driver.quit()
    print("\nBrowser closed successfully.")

**Compare notes with the instructor's live demo:** same clicks, same result — the only difference is you didn't see the window open.

---
## Part 6 — Capstone: Build the Full Pipeline ⏱ ~10 min
*(companion to slide: "Capstone: Build the Full Pipeline")*

Now we package what we did in Part 2.3 into one reusable **function** — a named recipe you can run again with different inputs:

**Input** → a search term (e.g. an RNA family name)
**Process** → query RNAcentral, build a table, clean it
**Output** → a research-ready CSV

The function is mostly written for you. Notice the same two patterns from Part 2: *request → `.json()`* and *list of dictionaries → DataFrame*.

ℹ️ Running this cell only **defines** the function — it prints nothing. We use it in the next cell.

In [ ]:
def build_rna_dataset(search_term, page_size=25, taxon_id=9606):
    # Search RNAcentral for `search_term`, clean the results, return a tidy DataFrame.
    url = "https://rnacentral.org/api/v1/rna"
    params = {
        "query": f"{search_term} AND TAXONOMY:{taxon_id}",
        "format": "json",
        "page_size": page_size,
    }

    # 1) request the data — same as Part 2.3
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    results = resp.json().get("results", [])

    # 2) one dictionary per record — same as Part 2.3
    # ✏️ TODO: fill in the two blanks, exactly like you did in Part 2.3
    # 💡 Hint: r.get("length")  and  r.get("rna_type")
    rows = []
    for r in results:
        rows.append({
            "rnacentral_id": r.get("rnacentral_id"),
            "description": r.get("description"),
            "length": ___,
            "rna_type": ___,
        })
    df = pd.DataFrame(rows)

    # 3) clean: remove duplicates, drop rows missing key info, label unknown types
    df = df.drop_duplicates(subset="rnacentral_id")
    df = df.dropna(subset=["rnacentral_id", "length"])
    df["rna_type"] = df["rna_type"].fillna("unknown")
    return df.reset_index(drop=True)

print("Function build_rna_dataset is ready to use.")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
"length": r.get("length"),
"rna_type": r.get("rna_type"),
```

</details>

Now **use** the function. One line does the whole pipeline.

In [ ]:
final_dataset = build_rna_dataset("miRNA precursor", page_size=30)
print(f"Retrieved {len(final_dataset)} clean records")
final_dataset.head(10)

In [ ]:
final_dataset.to_csv("session1_rna_dataset.csv", index=False)
print("Saved session1_rna_dataset.csv")

# To download the file to your own computer, remove the # from the next two lines and run again:
# from google.colab import files
# files.download("session1_rna_dataset.csv")

---
## ⭐ Optional — Part 7: Extract Text & Tables from a Paper with Docling ⏱ ~10 min
*(companion to slide: "Extract Text & Tables from a Paper")*

> **Optional:** Docling downloads several AI models the first time it runs, so this part can take **3–5 minutes** on Colab. Great to try after the workshop. Nothing later depends on it.

Not all biological knowledge lives in a database — much of it is locked inside PDFs. **Docling** turns a PDF into structured, machine-readable text and tables.

We'll use a well-known, stable PDF as a demo: "Attention Is All You Need", the paper that introduced the Transformer (which comes up again in Session 3). You can swap in any paper URL from your own research.

In [ ]:
!pip install -q docling

pdf_url = "https://arxiv.org/pdf/1706.03762"
pdf_path = "sample_paper.pdf"

resp = requests.get(pdf_url, timeout=60)
resp.raise_for_status()
with open(pdf_path, "wb") as f:      # "wb" = write the raw bytes of the PDF to a file
    f.write(resp.content)

print(f"Downloaded {pdf_path} ({len(resp.content) / 1024:.0f} KB)")

In [ ]:
try:
    from docling.document_converter import DocumentConverter

    converter = DocumentConverter()
    # ✏️ TODO: convert our PDF. Replace the blank with the variable holding the PDF's file name.
    # 💡 Hint: pdf_path
    result = converter.convert(___)

    markdown_text = result.document.export_to_markdown()
    print(markdown_text[:2000])        # show the first 2000 characters
except Exception as e:
    print(f"Docling failed to run here ({e}) — falling back to basic text extraction")
    import pypdf
    reader = pypdf.PdfReader(pdf_path)
    markdown_text = "\n\n".join(page.extract_text() or "" for page in reader.pages)
    print(markdown_text[:2000])

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
result = converter.convert(pdf_path)
```

</details>

Docling also recognises **tables** in the PDF and can hand each one back as a DataFrame.

In [ ]:
try:
    tables = result.document.tables
    print(f"Found {len(tables)} table(s) in the document")
    if tables:
        first_table_df = tables[0].export_to_dataframe(doc=result.document)
        display(first_table_df.head(10))
except NameError:
    print("Docling did not run in the previous cell — table extraction needs the full Docling path")

🧪 **Try it yourself:** replace `pdf_url` with a paper relevant to your own RNA research — a protocol, a supplementary table, or a review article — and re-run the cells above.

---
## Wrap-Up

**What you built today:**
- A FASTA parser and a DataFrame built from its records
- Working queries against the NCBI, UniProt and RNAcentral APIs — and a look at their docs
- A decoded RCSB PDB search request, sent straight from Python
- A BeautifulSoup scraper turning an HTML table into clean data
- A reusable data pipeline saved as `session1_rna_dataset.csv`
- ⭐ *(optional)* a headless Selenium automation and PDF text/table extraction with Docling

**Key pattern to remember:** *request → `.json()` → list of dictionaries → `pd.DataFrame` → `.to_csv()`*

**Next session:** we use data like this with scikit-learn to build a coding vs. non-coding RNA classifier.